# Import libraries

In [5]:
import os
import pandas as pd
import utils_emotions as ue
import utils_llm_filtering as ulf

In [6]:
INPUT_PATH = "../data/v2/clean_posts_bsky.csv"
RESULTS_PATH = "../data/v2/"

# Bart filtering

In [ ]:
# executed on cluster
# filtered_df = pd.read_csv(os.path.join(RESULTS_PATH, "bsky_emotion_filtered_posts.csv"))
# classifier = ulf.load_zero_shot_classifier(
#     model_name="facebook/bart-large-mnli",
#     device=-1  # use 0 if you have GPU
# )

# classified_df = ulf.add_llm_filter_columns(
#     filtered_df,
#     classifier=classifier,
#     text_column="Clean_Comment",
#     progress_every=100
# )

In [8]:
classified_df = pd.read_csv(os.path.join(RESULTS_PATH, "llm_classified_posts_bsky.csv"))
ai_df, review_df = ulf.filter_ai_related_comments(
    classified_df,
    min_score=0.50,
    min_margin=0.05,
    keep_uncertain_ai=True
)

os.makedirs(RESULTS_PATH, exist_ok=True)
ai_df.to_csv(os.path.join(RESULTS_PATH, "llm_filtered_ai_posts_bsky.csv"), index=False)
review_df.to_csv(os.path.join(RESULTS_PATH, "llm_review_posts_bsky.csv"), index=False)

Kept 3,140 AI-related comments.
Flagged 9,228 comments for rejection or manual review.


In [9]:
# how many posts in ai_df and review_df
print(f"Number of AI-related posts: {len(ai_df)}")
print(f"Number of posts for review: {len(review_df)}")

Number of AI-related posts: 3140
Number of posts for review: 9228


In [11]:
# After prompt reviewing of the review_df
after_review_df = pd.read_csv(os.path.join(RESULTS_PATH, "llm_review_related_ai_misclassified.csv"))
print(f"Number of posts after review: {len(after_review_df)}")

Number of posts after review: 7609


In [14]:
# Concat ai_df and after_review_df
final_ai_df = pd.concat([ai_df, after_review_df], ignore_index=True, sort=False)

# Some borderline AI rows are present in both files because filter_ai_related_comments
# keeps uncertain AI rows while also appending them to review_df for auditing.
if "Post_ID" in final_ai_df.columns:
    duplicate_posts = final_ai_df["Post_ID"].duplicated().sum()
    final_ai_df = final_ai_df.drop_duplicates(subset=["Post_ID"], keep="last").reset_index(drop=True)
    print(f"Dropped {duplicate_posts:,} duplicate Post_ID rows after joining.")

print(f"Final AI-related posts: {len(final_ai_df):,}")
print(f"Final columns: {len(final_ai_df.columns)}")

Dropped 3,120 duplicate Post_ID rows after joining.
Final AI-related posts: 7,629
Final columns: 30


In [13]:
# Save the joined AI-related DataFrame to a new CSV file
final_ai_df.to_csv(
    os.path.join(RESULTS_PATH, "bsky_AI_filtered_posts.csv"),
    index=False
)